# Preprocessing for modelling: test shop + Sets A–D

This notebook turns the raw sets into **one weekly panel** and then into a **feature frame** for a global forecasting model (e.g. LightGBM with L1 loss), following the playbook in [DATASETS.md](DATASETS.md) §8. The findings it relies on come from [set_a_b_test_analysis.ipynb](set_a_b_test_analysis.ipynb) and [set_c_d_analysis.ipynb](set_c_d_analysis.ipynb).

**How it is organised**
- Every operation is a small function `f(df, **options) -> df`.
- A pipeline is just a **list of steps**: `[func, (func, {options}), ...]`. `run_pipeline` applies them in order and logs the shape after each step.
- **To change the preprocessing, edit only the lists in §6 (per source) and §8 (features):** reorder entries, comment them out, or change their options. The functions above them don't need to change.

| § | What |
|---|---|
| 1 | Setup and the pipeline runner |
| 2 | Loaders (fix data types, one per set) |
| 3 | Source-specific cleaning steps |
| 4 | Daily → weekly aggregation and the common panel format |
| 5 | Panel cleaning steps shared by all sources |
| 6 | **Configure the per-source pipelines** and build the panel |
| 7 | Feature functions (lags, rolling stats, recency, intermittency, seasonality, targets, scaling) |
| 8 | **Configure the feature pipeline** and build the feature frame |
| 9 | Train / validation / prediction sets |
| 10 | Sanity check: the median baseline from the feature frame must match the EDA backtest |
| 11 | Save |

**Panel format** (output of §6): one row per `series_id × week_start`, a complete Monday grid per series with no gaps.

| column | meaning |
|---|---|
| `source` | `test_shop`, `A`, `B`, `C` or `D` |
| `series_id` | `source:id`, e.g. `A:61\|124` (store \| article) or `D:PROD-139867` |
| `week_start` | Monday of the week |
| `units` | units sold. **`NaN` = unknown** (stock-out week, not listed, partial week), never "no sale" |
| `censored` | the week's sales may have been capped by stock |
| `store_id`, `article_id` (+ optional hierarchy columns) | for reference / static features |

## 1. Setup and the pipeline runner

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

# Works whether the kernel starts in the notebook folder or the project root
BASE = Path('.') if Path('set-a').exists() else Path('data/hackathon-retail-sets')
OUT = BASE / 'processed'

H = 13                              # forecast horizon (w1..w13)
PAIR = ['store_id', 'article_id']


def _shape(obj):
    if isinstance(obj, pd.DataFrame):
        return f'{len(obj):,} rows x {obj.shape[1]} cols'
    if isinstance(obj, dict):
        return ', '.join(f'{k}: {len(v):,}' for k, v in obj.items())
    return type(obj).__name__


def run_pipeline(data, steps, name='', verbose=True):
    """Apply `steps` to `data` in order.

    Each step is either a function `f(data) -> data` or a tuple `(f, {options})`.
    `None` entries are skipped, so a step can be switched off with `cond and step`.
    """
    for step in steps:
        if step is None:
            continue
        func, kw = (step, {}) if callable(step) else (step[0], dict(step[1]) if len(step) > 1 else {})
        t0 = time.perf_counter()
        data = func(data, **kw)
        if verbose:
            print(f'[{name}] {func.__name__:<28s} -> {_shape(data):<26s} ({time.perf_counter() - t0:.1f}s)')
    return data


def to_week(dates):
    """Monday of the week (same convention as the test shop's week_start)."""
    return (dates - pd.to_timedelta(dates.dt.day_of_week, unit='D')).dt.normalize()

## 2. Loaders

Each loader reads one set, fixes data types and returns a single "fact" DataFrame. Nothing is filtered here, so any cleaning stays visible as a step in §6.

- **Test shop**: article × week, already a full grid.
- **Set A**: store × article × week, joined with the article hierarchy. `format` is dropped because it duplicates `store_format` (DATASETS §3).
- **Set B**: synthetic. `load_set_b_tables` returns every table with the known data issues fixed; `load_set_b` returns daily store × article sales lines for the weekly aggregation. Off by default (§6).
- **Set C**: numbers are `Decimal` and dates are `date` objects, so they are converted to float / datetime.
- **Set D**: sales lines (transactions only). The stock table has its own loader but isn't used by default (it is incomplete, DATASETS §6).

In [2]:
def load_test_shop(base=BASE):
    df = pd.read_parquet(base / 'test-shop' / 'history.parquet')
    df['week_start'] = pd.to_datetime(df['week_start'])
    df['on_hand'] = df['on_hand'].astype(float)
    return df.sort_values(['article_id', 'week_start'], ignore_index=True)


def load_set_a(base=BASE, with_articles=True, drop_cols=('format',)):
    df = pd.read_parquet(base / 'set-a' / 'weekly.parquet')
    df['week_start'] = pd.to_datetime(df['week_start'])
    if with_articles:
        art = pd.read_parquet(base / 'set-a' / 'articles.parquet').drop(columns=list(drop_cols))
        df = df.merge(art, on=PAIR, how='left', validate='many_to_one')
    return df.sort_values([*PAIR, 'week_start'], ignore_index=True)


def load_set_b_tables(base=BASE):
    """All Set B tables with the issues from DATASETS §4 fixed."""
    t = {p.stem: pd.read_parquet(p) for p in sorted((base / 'set-b').glob('*.parquet'))}
    t['baskets']['timestamp'] = pd.to_datetime(t['baskets']['timestamp'])
    t['stock_events']['timestamp'] = pd.to_datetime(t['stock_events']['timestamp'])
    for c in ['start_date', 'end_date']:
        t['promotions'][c] = pd.to_datetime(t['promotions'][c])
    t['stock']['store_id'] = t['stock']['store_id'].str.lstrip('S').astype(int)   # 'S0001' -> 1, like every other table
    t['stock_events'] = t['stock_events'].drop_duplicates(ignore_index=True)     # 181 exact duplicates
    return t


def load_set_b(base=BASE):
    """Daily store x article sales from the basket lines (transactions only, like Set D)."""
    t = load_set_b_tables(base)
    lines = t['sales_lines'].merge(t['baskets'][['transaction_id', 'store_id', 'timestamp']],
                                   on='transaction_id', how='left', validate='many_to_one')
    lines['date'] = lines['timestamp'].dt.normalize()
    daily = lines.groupby(['date', *PAIR], as_index=False)['units'].sum()
    return daily.sort_values([*PAIR, 'date'], ignore_index=True)


C_NUM = ['is_return', 'units', 'sales_amount', 'cogs', 'discount_amount', 'discount_pct', 'on_hand',
         'unit_cost', 'unit_price', 'latest_on_hand', 'days_since_receipt', 'days_since_sale']


def load_set_c(base=BASE):
    df = pd.read_parquet(base / 'set-c' / 'daily.parquet')
    df[C_NUM] = df[C_NUM].astype(float)
    df['is_return'] = df['is_return'].astype(int)
    for c in ['date', 'last_receipt_date', 'last_sold_date']:
        df[c] = pd.to_datetime(df[c])
    return df.sort_values([*PAIR, 'date'], ignore_index=True)


def load_set_d(base=BASE):
    df = pd.read_parquet(base / 'set-d' / 'sales.parquet')
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values([*PAIR, 'date'], ignore_index=True)


def load_set_d_stock(base=BASE):
    """Stock validity intervals. Open intervals (9999-12-31) are closed at the last sales date."""
    st = pd.read_parquet(base / 'set-d' / 'stock.parquet')
    end = pd.to_datetime(pd.read_parquet(base / 'set-d' / 'sales.parquet', columns=['date'])['date']).max()
    st['is_open'] = st['end_date'].eq('9999-12-31')
    st['start_date'] = pd.to_datetime(st['start_date'])
    st['end_date'] = pd.to_datetime(st['end_date'].where(~st['is_open'], str(end.date())))
    return st

## 3. Source-specific cleaning steps

These work on the raw fact tables from §2, before the weekly aggregation.

| Step | Used for | Why (DATASETS.md) |
|---|---|---|
| `ts_flag_censored` | test shop | 0 units **and** `on_hand ≤ 0` at the start of the week = possibly a stock-out (§8.7) |
| `a_flag_out_of_stock` | A | `in_stock = False` means sales were capped (§3) |
| `drop_pairs` | C | the PROD-0000202 burst at STR-022 (292 units in one day, §5) |
| `drop_columns` | C, D | leaky columns in C (`is_phasing_out`, same-day `days_since_sale`, …) and constant ones in D |
| `handle_returns` | C, D | net returns (default) or ignore them |
| `filter_sales_types` | C, D | optional: e.g. leave out clearance sales |
| `clean_prices` | C | price = 0 and price spikes → `NaN`. Only matters if you aggregate prices as features |

In [3]:
# ---------- test shop ----------
def ts_flag_censored(df, max_on_hand=0):
    """censored = no sale AND on_hand at the end of the previous week <= max_on_hand.
    on_hand is noisy (DATASETS §2), so this is only a hint. Masking is a separate step (mask_censored)."""
    prev = df.groupby('article_id')['on_hand'].shift(1)
    return df.assign(censored=(df['units'] == 0) & (prev <= max_on_hand))


# ---------- Set A ----------
def a_flag_out_of_stock(df):
    """in_stock=False -> the week's sales were capped by stock."""
    return df.assign(censored=~df['in_stock'].astype(bool))


# ---------- generic, used for C / D (and B) ----------
C_OUTLIER_PAIRS = [('STR-022', 'PROD-0000202')]
C_LEAKY_COLS = ['is_phasing_out', 'latest_on_hand', 'on_hand', 'days_since_sale', 'last_sold_date']
D_CONSTANT_COLS = ['sales_channel', 'store_type', 'return_reason']


def drop_pairs(df, pairs, keys=PAIR):
    """Drop whole series, e.g. pairs=[('STR-022', 'PROD-0000202')]."""
    bad = pd.MultiIndex.from_tuples(pairs, names=keys)
    return df[~pd.MultiIndex.from_frame(df[keys]).isin(bad)].reset_index(drop=True)


def drop_columns(df, cols):
    return df.drop(columns=[c for c in cols if c in df.columns])


def drop_constant_columns(df):
    return df.loc[:, df.nunique(dropna=False) > 1]


def handle_returns(df, mode='net'):
    """Returns are rows with is_return == 1 and negative units (C and D).
    'net'    : keep them, so weekly units = sold - returned (clip later with clip_negative).
    'ignore' : set their units to 0 (rows are kept, so Set C's listed days stay correct)."""
    if mode == 'net':
        return df
    if mode == 'ignore':
        return df.assign(units=df['units'].where(df['is_return'] != 1, 0.0))
    raise ValueError(mode)


def filter_sales_types(df, exclude=('CLEARANCE',)):
    """Set units to 0 for the given sales types (rows are kept). Set C codes: R, C, PROMO, MD (+ '_RETURN');
    Set D: FULL_PRICE, PROMO, MARKDOWN_TIER_1/2, CLEARANCE."""
    excl = set(exclude) | {f'{t}_RETURN' for t in exclude}
    return df.assign(units=df['units'].where(~df['sales_type'].isin(excl), 0.0))


def clean_prices(df, col='unit_price', spike_factor=10):
    """Price 0 and prices > spike_factor x the article's median price -> NaN (Set C glitches)."""
    p = df[col].where(df[col] > 0)
    med = p.groupby(df['article_id']).transform('median')
    return df.assign(**{col: p.mask(p > spike_factor * med)})

## 4. Daily → weekly, and the common panel format

`daily_to_weekly` handles the two ways the daily sets store "no sale":

- **`grid='rows'` (Set C)**: there is a row for every *listed* day (filler rows have 0 units). A missing row means "not listed", so it is **not** a zero. Weeks with fewer than `min_listed_days` listed days get `units = NaN`.
- **`grid='calendar'` (Sets D, B)**: only transactions are stored. Every day from the series' first sale to the end of the data counts as 0 when there is no row.

Both modes build a complete Monday grid per series and, by default, drop the partial first/last week of the dataset (C starts on a Sunday and ends on a Monday, D starts on a Sunday and ends on a Friday).

`to_panel` maps any weekly table to the panel format described at the top.

In [4]:
def _weekly_grid(span, keys):
    """Complete Monday grid per series from a DataFrame with 'min'/'max' week columns indexed by `keys`."""
    n = ((span['max'] - span['min']).dt.days // 7 + 1).to_numpy()
    grid = pd.DataFrame(index=span.index.repeat(n)).reset_index()
    offset = grid.groupby(keys, sort=False).cumcount().to_numpy()
    grid['week_start'] = np.repeat(span['min'].to_numpy(), n) + pd.to_timedelta(offset * 7, unit='D')
    return grid


def daily_to_weekly(df, keys, grid='rows', min_listed_days=1, drop_partial_edge_weeks=True,
                    date_col='date', static_cols=(), extra_aggs=None):
    """Aggregate daily rows to a complete weekly grid per series (see the text above).
    static_cols: per-series attributes to carry over (first value), e.g. the product hierarchy.
    extra_aggs: e.g. {'discount_pct': 'mean', 'unit_price': 'median'} to keep weekly price features."""
    d = df.assign(week_start=to_week(df[date_col]))
    first_day, last_day = d[date_col].min(), d[date_col].max()
    aggs = {'units': ('units', 'sum'), 'listed_days': (date_col, 'nunique')}
    aggs.update({c: (c, how) for c, how in (extra_aggs or {}).items()})
    w = d.groupby([*keys, 'week_start'], observed=True).agg(**aggs).reset_index()

    span = w.groupby(keys)['week_start'].agg(['min', 'max'])
    if grid == 'calendar':
        span['max'] = to_week(pd.Series([last_day])).iloc[0]
    out = _weekly_grid(span, keys).merge(w, on=[*keys, 'week_start'], how='left')
    if static_cols:
        out = out.merge(d.groupby(keys)[list(static_cols)].first().reset_index(), on=keys, how='left')

    if grid == 'rows':
        out['listed_days'] = out['listed_days'].fillna(0)
        out['units'] = out['units'].where(out['listed_days'] >= min_listed_days)
    elif grid == 'calendar':
        out['units'] = out['units'].fillna(0.0)
        week_end = out['week_start'] + pd.Timedelta(days=6)
        out['listed_days'] = ((week_end.clip(upper=last_day) - out['week_start'].clip(lower=first_day)).dt.days + 1)
    else:
        raise ValueError(grid)

    if drop_partial_edge_weeks:
        out = out[(out['week_start'] >= first_day) & (out['week_start'] + pd.Timedelta(days=6) <= last_day)]
    return out.reset_index(drop=True)


def to_panel(df, source, id_cols, static_cols=()):
    """Standardise a weekly table to the panel format (see the top of the notebook)."""
    sid = df[id_cols[0]].astype(str)
    for c in id_cols[1:]:
        sid = sid + '|' + df[c].astype(str)
    out = pd.DataFrame({
        'source': source,
        'series_id': source + ':' + sid,
        'week_start': df['week_start'].to_numpy(),
        'units': df['units'].astype(float).to_numpy(),
        'censored': df['censored'].fillna(False).astype(bool).to_numpy() if 'censored' in df else False,
        'store_id': df['store_id'].astype(str).to_numpy() if 'store_id' in df else None,
        'article_id': df['article_id'].astype(str).to_numpy(),
    })
    for c in static_cols:
        out[c] = df[c].astype(str).to_numpy()
    return out.sort_values(['series_id', 'week_start'], ignore_index=True)

## 5. Panel cleaning steps (shared by all sources)

| Step | What it does |
|---|---|
| `clip_negative` | weekly net units < 0 (more returns than sales) → 0 |
| `mask_censored` | `units = NaN` where `censored`, so the week is neither a target nor a zero in the features |
| `handle_outliers` | robust z-score per series (median / MAD): flag, cap or mask extreme weeks. *Uses the whole series, so it peeks at the future a little.* |
| `trim_to_first_sale` | drop the weeks before a series' first sale (they aren't demand history). Series that never sold disappear |
| `drop_short_series` | drop series with too few weeks or too few sale weeks (meant for the extra sets, not the test shop) |
| `check_panel` | assert the panel format: unique keys and a gap-free weekly grid per series |

In [5]:
def clip_negative(panel, col='units'):
    return panel.assign(**{col: panel[col].clip(lower=0)})


def mask_censored(panel):
    return panel.assign(units=panel['units'].where(~panel['censored']))


def handle_outliers(panel, z=20, mode='flag', min_units=10):
    """Robust z = (units - median) / (1.4826 * MAD) per series; MAD = 0 is replaced by 1.
    A week is an outlier if z > `z` and units >= `min_units`.
    mode: 'flag' (adds is_outlier), 'cap' (clip to median + z * 1.4826 * MAD) or 'nan' (mask)."""
    g = panel.groupby('series_id', sort=False)['units']
    med = g.transform('median')
    mad = (panel['units'] - med).abs().groupby(panel['series_id'], sort=False).transform('median').replace(0, 1)
    limit = med + z * 1.4826 * mad
    is_out = (panel['units'] > limit) & (panel['units'] >= min_units)
    if mode == 'flag':
        return panel.assign(is_outlier=is_out)
    if mode == 'cap':
        return panel.assign(units=panel['units'].mask(is_out, np.floor(limit)))
    if mode == 'nan':
        return panel.assign(units=panel['units'].mask(is_out))
    raise ValueError(mode)


def trim_to_first_sale(panel):
    started = (panel['units'].fillna(0) > 0).groupby(panel['series_id'], sort=False).cummax()
    return panel[started].reset_index(drop=True)


def drop_short_series(panel, min_weeks=26, min_sale_weeks=2):
    g = panel.groupby('series_id', sort=False)
    n_weeks = g['week_start'].transform('size')
    n_sales = (panel['units'] > 0).groupby(panel['series_id'], sort=False).transform('sum')
    return panel[(n_weeks >= min_weeks) & (n_sales >= min_sale_weeks)].reset_index(drop=True)


def check_panel(panel):
    assert not panel.duplicated(['series_id', 'week_start']).any(), 'duplicate series_id x week_start'
    assert panel['week_start'].dt.day_of_week.eq(0).all(), 'week_start must be a Monday'
    gaps = panel.groupby('series_id', sort=False)['week_start'].diff().dropna().dt.days
    assert gaps.eq(7).all(), 'weekly grid has gaps (or is not sorted)'
    return panel

## 6. Configure the per-source pipelines ← edit here

Each source has a `load` function and a list of `steps`. Reorder, comment out or change options freely. Set `'enabled': False` to leave a source out of the panel.

Defaults follow DATASETS §8.4:

| Source | Level | Preparation |
|---|---|---|
| Test shop | article × week | flag possibly-censored weeks (masking is commented out: test it in a backtest first, §8.7). **Not** trimmed to the first sale, so all 2,904 articles stay in the frame (see §10) |
| Set A | store × article × week | mask `in_stock = False` weeks; keep promo / Christmas peaks (outliers only flagged) |
| Set C | store × article × week | drop leaky columns and the PROD-0000202 burst, net returns, Monday weeks, only listed weeks |
| Set D | article × week (chain) | net returns, Monday weeks, only weeks after the first sale |
| Set B | store × article × week | **off**: synthetic and too smooth (§4) |

`COMMON_STEPS` run at the end of every pipeline; `EXTRA_STEPS` (trim to the first sale, drop short series) only for the extra sets.

In [6]:
A_HIERARCHY = ['product_group', 'division', 'department', 'department_group', 'store_format']
CD_HIERARCHY = ['division', 'category', 'subcategory', 'segment']

COMMON_STEPS = [
    clip_negative,
    check_panel,
]
EXTRA_STEPS = [
    trim_to_first_sale,
    (drop_short_series, {'min_weeks': 26, 'min_sale_weeks': 2}),
]

SOURCES = {
    'test_shop': {
        'enabled': True,
        'load': load_test_shop,
        'steps': [
            (ts_flag_censored, {'max_on_hand': 0}),
            (to_panel, {'source': 'test_shop', 'id_cols': ['article_id']}),
            # mask_censored,                       # try in a backtest before switching on
            *COMMON_STEPS,
            # trim_to_first_sale,                  # keep OFF for the test shop, see §10
        ],
    },
    'A': {
        'enabled': True,
        'load': load_set_a,
        'steps': [
            a_flag_out_of_stock,
            (to_panel, {'source': 'A', 'id_cols': PAIR, 'static_cols': A_HIERARCHY}),
            mask_censored,                          # in_stock=False -> units unknown
            (handle_outliers, {'z': 20, 'mode': 'flag'}),   # 'cap' / 'nan' once the dept-30 rows are checked
            *COMMON_STEPS,
            *EXTRA_STEPS,
        ],
    },
    'C': {
        'enabled': True,
        'load': load_set_c,
        'steps': [
            (drop_pairs, {'pairs': C_OUTLIER_PAIRS}),
            (drop_columns, {'cols': C_LEAKY_COLS}),
            (handle_returns, {'mode': 'net'}),
            # (filter_sales_types, {'exclude': ['C']}),     # e.g. leave out clearance-status sales
            # clean_prices,                                  # only needed with extra_aggs on prices
            (daily_to_weekly, {'keys': PAIR, 'grid': 'rows', 'min_listed_days': 7,
                               'static_cols': CD_HIERARCHY[1:]}),
            (to_panel, {'source': 'C', 'id_cols': PAIR, 'static_cols': CD_HIERARCHY[1:]}),   # one division only
            *COMMON_STEPS,
            *EXTRA_STEPS,
        ],
    },
    'D': {
        'enabled': True,
        'load': load_set_d,
        'steps': [
            (drop_columns, {'cols': D_CONSTANT_COLS}),
            (handle_returns, {'mode': 'net'}),
            # (filter_sales_types, {'exclude': ['CLEARANCE']}),
            (daily_to_weekly, {'keys': ['article_id'], 'grid': 'calendar',
                               'static_cols': ['supplier_id', *CD_HIERARCHY]}),
            (to_panel, {'source': 'D', 'id_cols': ['article_id'], 'static_cols': ['supplier_id', *CD_HIERARCHY]}),
            *COMMON_STEPS,
            trim_to_first_sale,
            (drop_short_series, {'min_weeks': 8, 'min_sale_weeks': 2}),   # D has < 1 year of history
        ],
    },
    'B': {
        'enabled': False,
        'load': load_set_b,
        'steps': [
            (daily_to_weekly, {'keys': PAIR, 'grid': 'calendar'}),
            (to_panel, {'source': 'B', 'id_cols': PAIR}),
            *COMMON_STEPS,
            *EXTRA_STEPS,
        ],
    },
}


def build_panel(sources=SOURCES, verbose=True):
    panels = {}
    for name, cfg in sources.items():
        if not cfg.get('enabled', True):
            continue
        t0 = time.perf_counter()
        raw = cfg['load']()
        if verbose:
            print(f'[{name}] {"load":<28s} -> {_shape(raw):<26s} ({time.perf_counter() - t0:.1f}s)')
        panels[name] = run_pipeline(raw, cfg['steps'], name=name, verbose=verbose)
    return panels

In [7]:
panels = build_panel()
panel = check_panel(pd.concat(panels.values(), ignore_index=True))
panel['source'] = panel['source'].astype('category')
print(f'\nPanel: {len(panel):,} rows, {panel.series_id.nunique():,} series')

[test_shop] load                         -> 415,272 rows x 4 cols      (0.4s)
[test_shop] ts_flag_censored             -> 415,272 rows x 5 cols      (0.0s)
[test_shop] to_panel                     -> 415,272 rows x 7 cols      (0.3s)
[test_shop] clip_negative                -> 415,272 rows x 7 cols      (0.0s)
[test_shop] check_panel                  -> 415,272 rows x 7 cols      (0.2s)
[A] load                         -> 98,835 rows x 10 cols      (0.1s)
[A] a_flag_out_of_stock          -> 98,835 rows x 11 cols      (0.0s)
[A] to_panel                     -> 98,835 rows x 12 cols      (0.3s)
[A] mask_censored                -> 98,835 rows x 12 cols      (0.0s)
[A] handle_outliers              -> 98,835 rows x 13 cols      (0.0s)
[A] clip_negative                -> 98,835 rows x 13 cols      (0.0s)
[A] check_panel                  -> 98,835 rows x 13 cols      (0.1s)
[A] trim_to_first_sale           -> 91,701 rows x 13 cols      (0.0s)
[A] drop_short_series            -> 91,701 rows x 

### Check the panel against DATASETS §7

Zero weeks and demand classes (Syntetos–Boylan) per source, counted over the observed (non-`NaN`) weeks from the first sale. They should be close to the table in DATASETS §7. Small differences come from the cleaning choices above (masked weeks, short series dropped, partial weeks removed).

In [8]:
def sb_label(adi, cv2):
    """Syntetos-Boylan demand class from ADI (avg. weeks between sales) and CV² of non-zero sizes."""
    return np.select([(adi < 1.32) & (cv2 < 0.49), (adi >= 1.32) & (cv2 < 0.49), adi < 1.32],
                     ['smooth', 'intermittent', 'erratic'], 'lumpy')


def series_summary(panel):
    u = panel['units']
    g = panel.assign(obs=u.notna(), nz=u > 0, nzu=u.where(u > 0)).groupby(['source', 'series_id'], observed=True)
    st = g.agg(weeks=('week_start', 'size'), obs=('obs', 'sum'), nz=('nz', 'sum'), cens=('censored', 'sum'),
               mean_nz=('nzu', 'mean'), std_nz=('nzu', 'std'))
    st['adi'] = st.obs / st.nz
    st['cv2'] = (st.std_nz / st.mean_nz).pow(2).fillna(0)
    st['class'] = sb_label(st.adi, st.cv2)
    rows = {}
    for src, s in st.groupby(level='source', observed=True):
        rows[src] = {'series': len(s), 'rows': int(s.weeks.sum()), 'median_weeks': s.weeks.median(),
                     'unknown_weeks_%': round(100 * (1 - s.obs.sum() / s.weeks.sum()), 1),
                     'censored_%': round(100 * s.cens.sum() / s.weeks.sum(), 1),
                     'zero_weeks_%': round(100 * (1 - s.nz.sum() / s.obs.sum()), 1),
                     'median_units_sale_week': round(s.mean_nz.median(), 2),
                     **{f'{c}_%': round(100 * (s['class'] == c).mean(), 1)
                        for c in ['smooth', 'intermittent', 'erratic', 'lumpy']}}
    return pd.DataFrame(rows).T


display(series_summary(panel))

,series,rows,median_weeks,unknown_weeks_%,censored_%,zero_weeks_%,median_units_sale_week,smooth_%,intermittent_%,erratic_%,lumpy_%
A,599.0,91701.0,164.0,4.4,4.4,35.9,2.03,13.7,47.4,9.2,29.7
C,837.0,79183.0,104.0,20.1,0.0,68.6,1.40,0.0,85.1,0.0,14.9
D,2326.0,84717.0,41.0,0.0,0.0,49.3,1.92,8.0,59.1,7.6,25.3
test_shop,2904.0,415272.0,143.0,0.0,20.8,64.7,2.06,14.3,67.6,3.5,14.6


## 7. Feature functions

All features are computed per series on the panel and use **only information up to and including the row's week**. Each row is a *forecast origin* `t`: the last known week. Its targets are the weeks `t+1 … t+13`.

Convention: **`lag_1` is the origin week itself** (the last value before w1), `lag_2` the week before, and so on. `lag_52` is the week 52 weeks before w1.

| Function | Adds | Notes |
|---|---|---|
| `add_scale` | `scale` | per-row series scale from history up to `t` (default: mean absolute week-to-week change, the MASE denominator). Leak-free because it is expanding |
| `add_lags` | `lag_k` | |
| `add_rolling` | `roll_{stat}_{w}` | stats: `mean`, `median`, `std`, `max`, or quantiles `q25`, `q75`, … (useful for intermittent series) |
| `add_zero_share` | `zero_share_{w}` | share of zero weeks among the observed weeks |
| `add_recency` | `weeks_since_sale`, `age_weeks` | "dead article" and "new article" signals |
| `add_intermittency` | `adi_{w}`, `cv2_{w}`, `demand_class` | rolling Syntetos–Boylan class |
| `add_seasonal` | `seasonal_h{h}`, `seas_ratio_52` | `seasonal_h{h}` = the same week last year for horizon `h`. Ratio = last year's next 13 weeks ÷ last year's previous 13 weeks. Lag-based only, because the test shop's dates are shifted (§8.5) |
| `add_targets` | `target_h{h}` | units in week `t+h` |
| `add_source_info` | `is_test_shop`, `sample_weight` | weight the test shop higher (§8.4) |
| `apply_scaling` | (in place) + `log_scale` | divides lags, rolling stats, seasonal values and targets by `scale` so a 494-unit week doesn't dominate 0/1 series. Remove it to train on raw units |
| `downcast` | | float64 → float32 to halve memory |

Order matters only for `add_scale` → `apply_scaling` (scale must exist) and `apply_scaling` must come after the columns it scales.

In [9]:
SID = 'series_id'


def _grp(panel, col):
    return panel.groupby(panel[SID], sort=False, observed=True)[col] if isinstance(col, str) else col.groupby(panel[SID], sort=False)


def _roll(panel, s, window, how='mean', **kw):
    """Grouped rolling (window=int) or expanding (window=None) statistic of Series `s`, aligned to panel."""
    g = s.groupby(panel[SID], sort=False)
    r = g.expanding(min_periods=1) if window is None else g.rolling(window, min_periods=1)
    res = r.quantile(kw['q']) if how == 'quantile' else getattr(r, how)()
    return res.reset_index(level=0, drop=True).sort_index()


def add_scale(panel, method='mase', window=None, col='units'):
    """method: 'mase' (mean |diff|), 'mean' (mean units) or 'nonzero_mean'. window=None -> expanding.
    Falls back to the non-zero mean, then 1, when the scale is 0 / undefined (e.g. the first week)."""
    u = panel[col]
    base = {'mase': _grp(panel, col).diff().abs(), 'mean': u, 'nonzero_mean': u.where(u > 0)}[method]
    scale = _roll(panel, base, window)
    fallback = _roll(panel, u.where(u > 0), window)
    scale = scale.where(scale > 0, fallback)
    return panel.assign(scale=scale.where(scale > 0, 1.0).fillna(1.0))


def add_lags(panel, lags=(*range(1, 14), 26, 52), col='units'):
    g = _grp(panel, col)
    return panel.assign(**{f'lag_{k}': g.shift(k - 1) for k in lags})


def add_rolling(panel, windows=(4, 8, 13, 26, 52), stats=('mean', 'median'), col='units'):
    new = {}
    for w in windows:
        for st in stats:
            if st.startswith('q'):
                new[f'roll_{st}_{w}'] = _roll(panel, panel[col], w, 'quantile', q=int(st[1:]) / 100)
            else:
                new[f'roll_{st}_{w}'] = _roll(panel, panel[col], w, st)
    return panel.assign(**new)


def add_zero_share(panel, windows=(13, 52), col='units'):
    is_zero = (panel[col] == 0).astype(float).where(panel[col].notna())
    return panel.assign(**{f'zero_share_{w}': _roll(panel, is_zero, w) for w in windows})


def add_recency(panel, col='units'):
    """weeks_since_sale: 0 if the origin week has a sale. age_weeks: weeks since the first sale."""
    sold = panel[col].fillna(0) > 0
    pos = panel.groupby(SID, sort=False, observed=True).cumcount().astype(float)
    last_sale = _grp(panel, pos.where(sold)).ffill()
    started = _grp(panel, sold).cummax()
    age = _grp(panel, started.astype(int)).cumsum() - 1
    return panel.assign(weeks_since_sale=pos - last_sale, age_weeks=age.where(started))


def add_intermittency(panel, window=52, col='units'):
    u = panel[col]
    nz = (u > 0).astype(float)
    x = u.where(u > 0, 0).fillna(0)
    n_nz, n_obs = _roll(panel, nz, window, 'sum'), _roll(panel, u.notna().astype(float), window, 'sum')
    s1, s2 = _roll(panel, x, window, 'sum'), _roll(panel, x ** 2, window, 'sum')
    adi = (n_obs / n_nz).where(n_nz > 0)
    mean = s1 / n_nz
    var = ((s2 - n_nz * mean ** 2) / (n_nz - 1)).where(n_nz > 1, 0).clip(lower=0)
    cv2 = (var / mean ** 2).where(n_nz > 0)
    cls = pd.Series(sb_label(adi.fillna(np.inf), cv2.fillna(0)), index=panel.index).where(n_nz > 0, 'no_sales')
    return panel.assign(**{f'adi_{window}': adi, f'cv2_{window}': cv2,
                           'demand_class': pd.Categorical(cls, ['smooth', 'intermittent', 'erratic', 'lumpy', 'no_sales'])})


def add_seasonal(panel, horizon=H, col='units', ratio=True, smoothing=1.0):
    g = _grp(panel, col)
    new = {f'seasonal_h{h}': g.shift(52 - h) for h in range(1, horizon + 1)}
    if ratio:
        r13 = _roll(panel, panel[col], 13)
        nxt = _grp(panel, r13).shift(52 - 13)   # mean of weeks t-51 .. t-39 = last year's w1..w13
        prv = _grp(panel, r13).shift(52)        # mean of weeks t-64 .. t-52 = last year's 13 weeks before
        new['seas_ratio_52'] = (nxt + smoothing) / (prv + smoothing)
    return panel.assign(**new)


def add_targets(panel, horizon=H, col='units'):
    g = _grp(panel, col)
    return panel.assign(**{f'target_h{h}': g.shift(-h) for h in range(1, horizon + 1)})


def add_source_info(panel, weights=None):
    """weights: e.g. {'test_shop': 3, 'A': 1, 'C': 1, 'D': 1}; missing sources get weight 1."""
    w = panel['source'].astype(str).map(weights or {}).fillna(1.0).astype(float)
    return panel.assign(is_test_shop=(panel['source'] == 'test_shop').astype(int), sample_weight=w)


SCALED_PREFIXES = ('lag_', 'roll_', 'seasonal_h', 'target_h')


def apply_scaling(panel, prefixes=SCALED_PREFIXES, scale_col='scale'):
    cols = [c for c in panel.columns if c.startswith(prefixes)]
    out = panel.copy()
    out[cols] = out[cols].div(out[scale_col], axis=0)
    out['log_scale'] = np.log(out[scale_col])
    return out


def downcast(panel):
    f64 = panel.select_dtypes('float64').columns
    return panel.astype({c: 'float32' for c in f64})

## 8. Configure the feature pipeline ← edit here

In [10]:
FEATURE_STEPS = [
    (add_scale, {'method': 'mase', 'window': None}),
    (add_lags, {'lags': (*range(1, 14), 26, 52)}),
    (add_rolling, {'windows': (4, 8, 13, 26, 52), 'stats': ('mean', 'median')}),
    (add_rolling, {'windows': (13, 26), 'stats': ('q25', 'q75', 'max')}),
    (add_zero_share, {'windows': (13, 52)}),
    add_recency,
    (add_intermittency, {'window': 52}),
    (add_seasonal, {'horizon': H, 'ratio': True}),
    (add_targets, {'horizon': H}),
    (add_source_info, {'weights': {'test_shop': 3.0}}),
    apply_scaling,                                   # comment out to keep raw units
    downcast,
]

frame = run_pipeline(panel, FEATURE_STEPS, name='features')
frame.head()

[features] add_scale                    -> 670,873 rows x 18 cols     (0.6s)
[features] add_lags                     -> 670,873 rows x 33 cols     (0.2s)
[features] add_rolling                  -> 670,873 rows x 43 cols     (4.5s)
[features] add_rolling                  -> 670,873 rows x 49 cols     (3.0s)
[features] add_zero_share               -> 670,873 rows x 51 cols     (0.9s)
[features] add_recency                  -> 670,873 rows x 53 cols     (0.3s)
[features] add_intermittency            -> 670,873 rows x 56 cols     (2.0s)
[features] add_seasonal                 -> 670,873 rows x 70 cols     (0.6s)
[features] add_targets                  -> 670,873 rows x 83 cols     (0.2s)
[features] add_source_info              -> 670,873 rows x 85 cols     (0.3s)
[features] apply_scaling                -> 670,873 rows x 86 cols     (0.6s)
[features] downcast                     -> 670,873 rows x 86 cols     (0.3s)


,source,series_id,week_start,units,censored,store_id,article_id,product_group,division,department,department_group,store_format,is_outlier,category,subcategory,segment,supplier_id,scale,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,lag_11,lag_12,lag_13,lag_26,lag_52,roll_mean_4,roll_median_4,roll_mean_8,roll_median_8,roll_mean_13,roll_median_13,roll_mean_26,...,roll_q25_26,roll_q75_26,roll_max_26,zero_share_13,zero_share_52,weeks_since_sale,age_weeks,adi_52,cv2_52,demand_class,seasonal_h1,seasonal_h2,seasonal_h3,seasonal_h4,seasonal_h5,seasonal_h6,seasonal_h7,seasonal_h8,seasonal_h9,seasonal_h10,seasonal_h11,seasonal_h12,seasonal_h13,seas_ratio_52,target_h1,target_h2,target_h3,target_h4,target_h5,target_h6,target_h7,target_h8,target_h9,target_h10,target_h11,target_h12,target_h13,is_test_shop,sample_weight,log_scale
0,test_shop,test_shop:A100181,2020-07-27,5.0,False,None,A100181,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,0.00,0.00,0.0,0.0,1.000000,0.000000,smooth,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.200000,0.2,0.00,0.200000,0.400000,0.000000,0.000000,0.200000,0.40,0.2,0.400000,0.200000,0.600000,1,3.0,1.609438
1,test_shop,test_shop:A100181,2020-08-03,1.0,False,None,A100181,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,0.250000,1.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.750000,0.750000,0.750000,0.750000,0.750000,0.750000,0.750000,...,0.500000,1.000000,1.250000,0.00,0.00,0.0,1.0,1.000000,0.888889,erratic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.250000,0.0,0.25,0.500000,0.000000,0.000000,0.250000,0.500000,0.25,0.5,0.250000,0.750000,0.000000,1,3.0,1.386294
2,test_shop,test_shop:A100181,2020-08-10,1.0,False,None,A100181,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,0.500000,0.50,2.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.166667,0.500000,1.166667,0.500000,1.166667,0.500000,1.166667,...,0.500000,1.500000,2.500000,0.00,0.00,0.0,2.0,1.000000,0.979592,erratic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.5,1.00,0.000000,0.000000,0.500000,1.000000,0.500000,1.00,0.5,1.500000,0.000000,0.500000,1,3.0,0.693147
3,test_shop,test_shop:A100181,2020-08-17,0.0,False,None,A100181,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.666667,0.000000,0.60,0.600000,3.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.050000,0.600000,1.050000,0.600000,1.050000,0.600000,1.050000,...,0.450000,1.200000,3.000000,0.25,0.25,1.0,3.0,1.333333,0.979592,lumpy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.600000,1.2,0.00,0.000000,0.600000,1.200000,0.600000,1.200000,0.60,1.8,0.000000,0.600000,0.600000,1,3.0,0.510826
4,test_shop,test_shop:A100181,2020-08-24,1.0,False,None,A100181,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.500000,0.666667,0.00,0.666667,0.666667,3.333333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500000,0.666667,1.066667,0.666667,1.066667,0.666667,1.066667,...,0.666667,0.666667,3.333333,0.20,0.20,0.0,4.0,1.250000,1.000000,erratic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.333333,0.0,0.00,0.666667,1.333333,0.666667,1.333333,0.666667,2.00,0.0,0.666667,0.666667,3.333333,1,3.0,0.405465


## 9. Train / validation / prediction sets

- The extra sets cover other shops and other years, so **all their origins** with known targets go into training.
- For the test shop, a backtest **cutoff** is the last known week. Training uses test-shop origins whose whole 13-week target window ends on or before the cutoff. Validation is the test-shop rows *at* the cutoff, scored on the next 13 weeks.
- Use several cutoffs (`rolling_cutoffs`): scores move ~25 points between windows (DATASETS §8.2).
- The prediction set for the submission is the test shop's last week (`cutoff=None`).

`to_long` turns the wide frame (one row per origin, `target_h1..h13`) into one row per origin × horizon with columns `h`, `target` and `seasonal_lag52` (the matching `seasonal_h{h}`), which is what a single direct model trains on. It keeps only the model features plus a few id columns (`source`, `series_id`, `article_id`, `week_start`, `scale`, `sample_weight`).

`subsample_origins` keeps every k-th origin per source (`ORIGIN_STRIDE`) and drops origins before a series' first sale. With the defaults the training frame is ~5.3M rows / ~1 GB. Raise the strides if memory is tight.

Predictions from a model trained on the scaled target must be multiplied back by `scale` before scoring or submitting.

In [11]:
META_COLS = ['source', SID, 'week_start', 'units', 'censored', 'store_id', 'article_id', 'scale', 'is_outlier', 'sample_weight',
             'supplier_id', *A_HIERARCHY, *CD_HIERARCHY]


def feature_columns(df):
    """Model inputs of a long frame: everything except meta columns and the target."""
    return [c for c in df.columns if c not in META_COLS and c not in ('target',) and not c.startswith('target_h')]


def rolling_cutoffs(frame, n_windows=4, step=H, horizon=H, source='test_shop'):
    """Backtest cutoffs, latest first: the last week with a full 13-week holdout, then every `step` weeks back."""
    last = frame.loc[frame['source'] == source, 'week_start'].max()
    return [last - pd.Timedelta(weeks=horizon + i * step) for i in range(n_windows)]


def split_frame(frame, cutoff=None, horizon=H, valid_source='test_shop'):
    """Returns (train, valid). cutoff=None -> valid is the prediction set at the last week."""
    is_v = frame['source'] == valid_source
    if cutoff is None:
        cutoff = frame.loc[is_v, 'week_start'].max()
    target_end = frame['week_start'] + pd.Timedelta(weeks=horizon)
    train = frame[~is_v | (target_end <= cutoff)]
    valid = frame[is_v & (frame['week_start'] == cutoff)]
    return train.reset_index(drop=True), valid.reset_index(drop=True)


def subsample_origins(frame, every=None, min_history_weeks=1):
    """Keep every k-th origin per series, counted back from the series' last week; every={'A': 2, ...}.
    min_history_weeks: drop origins with less history (age_weeks counts from the first sale)."""
    keep = pd.Series(True, index=frame.index)
    if every:
        k = frame['source'].astype(str).map(every).fillna(1).astype(int)
        from_end = frame.groupby(SID, sort=False, observed=True).cumcount(ascending=False)
        keep &= (from_end % k == 0)
    if 'age_weeks' in frame:
        keep &= frame['age_weeks'] >= min_history_weeks - 1
    return frame[keep].reset_index(drop=True)


LONG_ID_COLS = ['source', SID, 'article_id', 'week_start', 'scale', 'sample_weight']


def to_long(frame, horizon=H, drop_missing_target=True, id_cols=LONG_ID_COLS):
    """One row per origin x horizon. Keeps the model features plus `id_cols`; ids become categoricals."""
    feats = [c for c in frame.columns if c not in META_COLS and not c.startswith(('target_h', 'seasonal_h'))]
    base = [c for c in id_cols if c in frame.columns] + feats
    parts = []
    for h in range(1, horizon + 1):
        p = frame[base].copy()
        p['h'] = np.int8(h)
        if f'seasonal_h{h}' in frame:
            p['seasonal_lag52'] = frame[f'seasonal_h{h}'].to_numpy()
        if f'target_h{h}' in frame:
            p['target'] = frame[f'target_h{h}'].to_numpy()
        parts.append(p)
    long = pd.concat(parts, ignore_index=True)
    long[[SID, 'article_id']] = long[[SID, 'article_id']].astype('category')
    if drop_missing_target and 'target' in long:
        long = long[long['target'].notna()].reset_index(drop=True)
    return long

In [12]:
ORIGIN_STRIDE = {'test_shop': 1, 'A': 2, 'C': 2, 'D': 1}   # keep every k-th origin per source in training

cutoffs = rolling_cutoffs(frame, n_windows=4)
print('Backtest cutoffs (last known week):', [c.date() for c in cutoffs])

train_w, valid_w = split_frame(frame, cutoff=cutoffs[0])
train = to_long(subsample_origins(train_w, every=ORIGIN_STRIDE))
valid = to_long(valid_w, drop_missing_target=False)     # keep every article; score on the raw units (§10)
FEATURES = feature_columns(train)

print(f'train: {len(train):,} rows ({train.source.value_counts().to_dict()})')
print(f'valid: {len(valid):,} rows = {valid_w[SID].nunique():,} test-shop articles x {H} horizons')
print(f'{len(FEATURES)} features:', FEATURES)

# Prediction set for the submission: the test shop's last week
_, predict_w = split_frame(frame, cutoff=None)
predict = to_long(predict_w, drop_missing_target=False)
print(f'predict: {len(predict):,} rows = {predict_w[SID].nunique():,} articles x {H} (origin {predict_w.week_start.max().date()})')

Backtest cutoffs (last known week): [datetime.date(2023, 1, 16), datetime.date(2022, 10, 17), datetime.date(2022, 7, 18), datetime.date(2022, 4, 18)]
train: 5,344,774 rows ({'test_shop': 3534076, 'D': 890450, 'A': 544847, 'C': 375401})
valid: 37,752 rows = 2,904 test-shop articles x 13 horizons
43 features: ['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'lag_13', 'lag_26', 'lag_52', 'roll_mean_4', 'roll_median_4', 'roll_mean_8', 'roll_median_8', 'roll_mean_13', 'roll_median_13', 'roll_mean_26', 'roll_median_26', 'roll_mean_52', 'roll_median_52', 'roll_q25_13', 'roll_q75_13', 'roll_max_13', 'roll_q25_26', 'roll_q75_26', 'roll_max_26', 'zero_share_13', 'zero_share_52', 'weeks_since_sale', 'age_weeks', 'adi_52', 'cv2_52', 'demand_class', 'seas_ratio_52', 'is_test_shop', 'log_scale', 'h', 'seasonal_lag52']
predict: 37,752 rows = 2,904 articles x 13 (origin 2023-04-17)


## 10. Sanity check: median baseline from the feature frame

If the alignment of lags, scale and targets is right, "forecast `roll_median_13`, flat for 13 weeks" from the feature frame must score like the EDA's *median last 13 w* baseline (−6.7 on the last window, DATASETS §2). `mase_score` and `make_submission` are the EDA notebook's versions, extended to accept forecasts for only some articles (articles with no sale before the cutoff are absent from the frame and get 0).

They should match exactly on every window.

**Why the test shop is not trimmed to its first sale (§6):** the MASE denominator is computed over the article's *whole* history, including the zeros before its launch. New articles therefore have tiny denominators. If their series start at the first sale, their rolling medians skip those zeros, come out higher, and the over-forecast costs about 20 points on this check. Keep the test-shop grid complete. Training still skips origins before an article's first sale (`subsample_origins(min_history_weeks=1)`).

In [13]:
ts_raw = load_test_shop()
Y = ts_raw.pivot(index='article_id', columns='week_start', values='units').sort_index(axis=1)


def forecast_matrix(df, pred_col, index=Y.index, horizon=H):
    """Long rows (article_id, h, prediction) -> articles x horizon array; missing articles -> 0."""
    F = df.pivot_table(index='article_id', columns='h', values=pred_col, aggfunc='first', observed=True)
    return F.reindex(index=index, columns=range(1, horizon + 1)).fillna(0).to_numpy(float)


def mase_score(cutoff, F, Y=Y, horizon=H):
    """Score (1 - MASE) x 100 of forecast matrix F (articles x horizon) for the 13 weeks after `cutoff`."""
    hist = Y.loc[:, :cutoff]
    act = Y.loc[:, Y.columns > cutoff].iloc[:, :horizon]
    den = hist.diff(axis=1).abs().mean(axis=1).to_numpy()
    err = np.abs(act.to_numpy() - np.clip(F, 0, None))
    out = {}
    with np.errstate(divide='ignore', invalid='ignore'):
        for name, cols in [('all w1-w13', slice(0, horizon)), ('w1-w7 (live)', slice(0, 7)), ('w8-w13 (final)', slice(7, horizon))]:
            m = err[:, cols].mean(axis=1) / den
            m = m[np.isfinite(m)]
            out[name] = round((1 - m.mean()) * 100, 1)
    return out


def make_submission(F, index=Y.index):
    F = np.clip(np.asarray(F, dtype=float), 0, None)
    sub = pd.DataFrame(F, index=index, columns=[f'w{i}' for i in range(1, H + 1)]).reset_index()
    assert len(sub) == 2904 and sub.article_id.is_unique and not sub.isna().any().any()
    return sub


scaled = 'log_scale' in frame
rows = []
for c in cutoffs:
    _, v = split_frame(frame, cutoff=c)
    v = to_long(v, drop_missing_target=False)
    v['pred'] = v['roll_median_13'] * (v['scale'] if scaled else 1)
    eda = np.repeat(Y.loc[:, :c].iloc[:, -13:].median(axis=1).to_numpy()[:, None], H, axis=1)
    rows.append({'cutoff': c.date(), **{f'frame {k}': s for k, s in mase_score(c, forecast_matrix(v, 'pred')).items()},
                 'EDA median 13w (all)': mase_score(c, eda)['all w1-w13']})
display(pd.DataFrame(rows).set_index('cutoff'))

# Target alignment: de-scaled target_h1 must equal the raw units of the week after the cutoff
v = valid.assign(raw=valid['target'] * (valid['scale'] if scaled else 1))
chk = v[v.h == 1].set_index(v.loc[v.h == 1, 'article_id'].astype(str))['raw']
nxt = Y.loc[chk.index, Y.columns[Y.columns > cutoffs[0]][0]]
print('target_h1 matches the raw units of week cutoff+1:', np.allclose(chk.to_numpy(), nxt.to_numpy(), atol=1e-3))

,frame all w1-w13,frame w1-w7 (live),frame w8-w13 (final),EDA median 13w (all)
cutoff,,,,
2023-01-16,-6.7,-6.5,-6.9,-6.7
2022-10-17,-30.5,-36.3,-23.8,-30.5
2022-07-18,-12.6,-7.2,-18.9,-12.6
2022-04-18,-16.3,-12.4,-21.0,-16.3


target_h1 matches the raw units of week cutoff+1: True


## 11. Save

Writes the panel and the wide feature frame to `processed/`. The train / validation / prediction sets are cheap to rebuild from the frame with §9, so they aren't saved.

In [14]:
SAVE = True
if SAVE:
    OUT.mkdir(exist_ok=True)
    panel.to_parquet(OUT / 'panel.parquet', index=False)
    frame.to_parquet(OUT / 'features_wide.parquet', index=False)
    for f in ['panel.parquet', 'features_wide.parquet']:
        print(f'{OUT / f}: {(OUT / f).stat().st_size / 1e6:.1f} MB')

processed\panel.parquet: 0.6 MB
processed\features_wide.parquet: 61.4 MB
